# Project 03: Gibbs Sampling (Random Algorithm)

In [ ]:
"""
project03.ipynb

This program implements a Gibbs Sampling algorithm to discover a short, shared DNA motif
across a list of sequences without knowing in advance where that motif falls
within each one. GibbsMotifFinder function builds an initial random guess for
each sequence, then repeatedly holds out one sequence at a time, builds a
consensus (a Position Frequency Matrix and Position Weight Matrix) from every
other sequence's current guess, scores every possible position in the held-out
sequence on both the forward and reverse-complement strands, and updates that
sequence's guess via weighted random selection favoring higher-scoring
positions, without ever being locked into always picking the single best
score. After a fixed number of rounds, a final consensus is built from every
sequence's last guess and returned as a Position Frequency Matrix.

For the Promotor Driver section, the function is tested on a
B. subtilis genome, pre-filtered down to sequences containing part of the
Shine-Dalgarno motif (AGGAGG), with each promoter sequence extracted using
get_seq from the 50 bp immediately upstream of each coding sequence (CDS)
identified from the accompanying GFF annotation file.

For the NRF1 Driver section, the same function is tested on a
second, larger dataset, real NRF1 ChIP-seq peak sequences (data/nrf1_gibbs.fa),
read in directly via get_fasta with no additional filtering, since every
sequence in that file is already a confirmed binding-site peak.

Authors are Julianne Murthy, Graziano Peregrino, and Ildiko Polyak.
"""

In [1]:
import random
import numpy as np
import seqlogo 
#import function for building sequence motif & idenfitying seqs matching to motif
from data_readers import *
from seq_ops import get_seq
from motif_ops import *

---
# Project Start

In [ ]:
def GibbsMotifFinder (seqs, k, seed=None):
    '''
        Function to find a pfm from a list of strings using a Gibbs sampler
        
        Args: 
            seqs (str list): a list of sequences, not necessarily in same lengths
            k (int): the length of motif to find
            seed (int, default=None): seed for np.random
    
        Returns:
            pfm (numpy array): dimensions are 4xlength
        '''

    max_iter = 10000

    # Use rng to make random samples/selections/numbers
    # Example: randint = rng.integer(1, 10)
    random.seed(seed)
    rng = np.random.default_rng(seed)
    total_seqs = len(seqs) #N, total number of seqs

 
    # INITIALIZE random valid start position per sequence
    motifs = []
    for seq in seqs:
        start_pos = rng.integers(0, len(seq) - k + 1) #picks a start position at random
        motifs.append(seq[start_pos:start_pos + k]) # hold one k length chunk per seq

    #print(f"num sequences: {total_seqs}")
    #print(f"first 3 motifs: {motifs[:3]}")
    #print(f"motif lengths all == k? {all(len(m) == k for m in motifs)}")
 
    # save the best scoring version of 4xk frequency matrix
    final_ic = -np.inf 
    final_pfm = build_pfm(motifs, k)
    #print(f"initial pfm shape: {final_pfm.shape}")  # should be (4, k)
    #print(f"initial pfm:\n{final_pfm}")
    stall_count = 0 # count how many iterations in a row that have failed improving
 
    for j in range(max_iter):
        # hold out one random sequence i
        i = rng.integers(0, total_seqs)
 
        # build pfm/pwm from all OTHER current motifs
        other_motifs = motifs[:i] + motifs[i + 1:]
        pfm_others = build_pfm(other_motifs, k) 
        pwm = build_pwm(pfm_others)

        #if j % 500 == 0:
            #print(f"\n--- iteration {j} ---")
            #print(f"held-out seq index: {i}")
            #print(f"pwm shape: {pwm.shape}")
 
        # score every position in the held out sequence, both strands, keeping the better scoring chunk at each position
        seq_i = seqs[i]
        possible_pos = len(seq_i) - k + 1 # grabs all possible k-length start positions within the held out seq i

        best_scores = np.empty(possible_pos) #create numpy array of length possible_pos
        best_chunks = []
 
        for pos in range(possible_pos):
            forward_chunk = seq_i[pos:pos + k]
            reverse_chunk = reverse_complement(forward_chunk)
            forward_score = score_kmer(forward_chunk, pwm)
            reverse_score = score_kmer(reverse_chunk, pwm)
 
            if forward_score >= reverse_score:
                best_scores[pos] = forward_score
                best_chunks.append(forward_chunk)
            else:
                best_scores[pos] = reverse_score
                best_chunks.append(reverse_chunk)
 
        # undo log2, positive weights, normalize to probabilities
        weights = np.power(2.0, best_scores)
        probs = weights / weights.sum()

        #if j % 500 == 0:
            #print(f"num candidate positions: {possible_pos}")
            #print(f"best_scores range: {best_scores.min():.3f} to {best_scores.max():.3f}")
 
        # weighted random pick (NOT max), update motif i
        chosen_pos = rng.choice(possible_pos, p=probs)
        motifs[i] = best_chunks[chosen_pos]
 
        #if j % 500 == 0:
            #print(f"chosen position: {chosen_pos}, new motif_i: {motifs[i]}")

        # convergence tracking
        tolerance = 1e-6 # removes requirement for exact floating points for ic and final_ic to be equal
        full_pfm = build_pfm(motifs, k)
        ic = pfm_ic(full_pfm)
        if ic > final_ic + tolerance:
            final_ic = ic
            final_pfm = full_pfm
            stall_count = 0
        else:
            stall_count += 1

        #if j % 500 == 0:
            #print(f"ic: {ic:.4f}, final_ic: {final_ic:.4f}, stall_count: {stall_count}")

 
        #if stall_count >= 500:  
            #print(f"Converged early at iteration {j}")
            #break
 
    #print(f"\nfinal IC: {final_ic:.4f}")
    #print(f"final pfm:\n{final_pfm}")
    return final_pfm


In [ ]:
# Here we test your Gibbs sampler.
# You do not need to edit this or the section below. This is the Driver program

#read promoters, store in a list of strings
seq_file = "data/GCF_000009045.1_ASM904v1_genomic.fna"
gff_file = "data/GCF_000009045.1_ASM904v1_genomic.gff"

seqs = []

for name, seq in get_fasta(seq_file): # For each entry in our FASTA file
    for gff_entry in get_gff(gff_file): # For each entry in our GFF file
        if gff_entry.type == 'CDS': # If this is a coding sequence
            promoter_seq = get_seq(seq, gff_entry.start, gff_entry.end, gff_entry.strand, 50) # Extract 50 bp as a promoter
    
            """
            Because the gibbs sampling assumption is broken in just using promoters,
            and because it takes very long time to randomly progress through so many
            regions, for this example we will pre-filter for sequences that all contain
            part of the shine-dalgarno motif:
            """
            if "AGGAGG" in promoter_seq:
                seqs.append(promoter_seq)

---
# Driver Program
Don't change any of the code here. If you have completed the project by following the coding by contract, the following code should work.

In [ ]:
# Run the gibbs sampler:
promoter_pfm = GibbsMotifFinder(seqs,10 )

# Plot the final pfm that is generated: 
seqlogo.seqlogo(seqlogo.CompletePm(pfm = promoter_pfm.T))


---
# NRF1 Driver Program
Modify the code below and test your Gibbs Sampler.

In [ ]:
# Here we test your Gibbs sampler.
# You do not need to edit this or the section below. This is the Driver program

'''
Pseudocode: 
nrf1_file="<path/to/file.fa>"
nrf1_peaks = []

FOR name, seq in get_fasta(nrf1_file):
nrf1_peaks.append(seq)
    


'''

#read promoters, store in a list of strings
nrf1_file = "data/nrf1_gibbs.fa"

nrf1_peaks = []

for name, seq in get_fasta(nrf1_file):
    nrf1_peaks.append(seq)


# Run the gibbs sampler:
nrf1_pfm = GibbsMotifFinder(nrf1_peaks, 10)

# Plot the final pfm that is generated: 
#seqlogo.seqlogo(seqlogo.CompletePm(pfm = nrf1_pfm.T)) #Note: comment out to remove empty graph because of GhostScript

#sanity check our results without in case seqlogo in NRF1 Driver Program doesn't work
#Note: NRF1 has ~90,000 sequences (versus ~800 for promoters), so it will take a long time to run 10,000 rounds.  
#A quick 100 round test showed working code but a low IC score, which is expected for just a few rounds
#print(nrf1_pfm)
#print(pfm_ic(nrf1_pfm))